# Dated leveraged buyout

Model a cash-free, debt-free acquisition at the 2025 reporting close. The seller
receives the entry enterprise value. Upfront fees add to cash uses. Two loans
bring in cash below their contractual face, so the forecast roll-forward starts
from face while funding uses cash proceeds.

In [ ]:
import pandas as pd

from finstack_quant.core.money import Money
from finstack_quant.portfolio import mwr_xirr
from finstack_quant.statements import ModelBuilder
from finstack_quant.statements_analytics import (
    LboDebtFunding,
    LboEarnings,
    LboFunding,
    LboRequest,
    StatementValuationCut,
    add_roll_forward_with_opening,
    evaluate_lbo,
)

ENTRY_DATE, EXIT_DATE = "2025-12-31", "2028-12-31"

def usd(amount):
    return Money(amount, "USD")

builder = ModelBuilder("dated-buyout").periods("2025..2028", "2025").currency("USD")
builder.source_snapshot("training-borrower-2025-close")
builder.value_money("ebitda", [("2025", usd(100)), ("2028", usd(120))])
builder.availability_dates("ebitda", [("2025", ENTRY_DATE)])
builder.value_money("eligible_cash", [("2028", usd(40))])
builder.value("term_amortization", [("2025", 0.0), ("2026", 30.0), ("2027", 30.0), ("2028", 20.0)])
add_roll_forward_with_opening(builder, "term", [], ["term_amortization"], 420.0)
add_roll_forward_with_opening(builder, "junior", [], [], 100.0)
builder.as_money("closing_debt", "term_end + junior_end")
snapshot = builder.build()
builder = snapshot.to_builder()
for node in snapshot.nodes:
    if "2025" in (node.values or {}):
        builder.availability_dates(node.node_id, [("2025", ENTRY_DATE)])
model = builder.build()

lbo_cut = StatementValuationCut(ENTRY_DATE, ENTRY_DATE, "USD")
debt_sources = [
    LboDebtFunding(name="term", cash_proceeds=usd(405), opening_face=usd(420)),
    LboDebtFunding(name="junior", cash_proceeds=usd(95), opening_face=usd(100)),
]

def price_buyout(entry_multiple="8", exit_multiple="9", sponsor_commitment=None):
    funding = LboFunding(
        date=ENTRY_DATE,
        acquisition={"kind": "cash_free_debt_free"},
        transaction_fees=usd(20),
        closing_credits=usd(0),
        debt=debt_sources,
        sponsor_commitment=sponsor_commitment,
    )
    return evaluate_lbo(
        model,
        LboRequest(
            cut=lbo_cut, exit_valuation_date=EXIT_DATE,
            entry=LboEarnings(node="ebitda", period="2025", multiple=entry_multiple, basis="as_observed"),
            funding=funding,
            exit=LboEarnings(node="ebitda", period="2028", multiple=exit_multiple, basis="as_observed"),
            exit_debt_node="closing_debt", exit_cash_node="eligible_cash",
        ),
    )

## Reconcile sources and uses

The sponsor first sizes the equity contribution. Compare a shortfall with a
fully funded case. A sizing return assumes the sponsor contributes the required
amount. A funded return requires the authored commitments to reconcile.

In [ ]:
sized = price_buyout()
short = price_buyout(sponsor_commitment=usd(300))
funded = price_buyout(sponsor_commitment=sized.funding.required_sponsor_cash)

assert sized.entry_enterprise_value.amount == 800
assert sized.funding.cash_uses.amount == 820
assert sized.funding.debt_cash_proceeds.amount == 500
assert sized.funding.opening_debt_face.amount == 520
assert short.funding.funding_gap.amount == 20 and short.exit.funded_moic is None
assert funded.funding.funding_gap.amount == 0
assert funded.exit.net_debt.amount == 400
assert funded.exit.sponsor_proceeds.amount == 680
assert funded.exit.funded_moic == 2.125
irr = mwr_xirr([
    (ENTRY_DATE, -float(funded.funding.required_sponsor_cash.amount)),
    (EXIT_DATE, float(funded.exit.sponsor_proceeds.amount)),
])
print(f"Funded MOIC {funded.exit.funded_moic:.3f}x and dated equity IRR {irr:.2%}")

## Compare entry and exit multiples

Hold earnings, debt proceeds and contractual amortization fixed. Reprice a
rectangular grid of purchase and exit multiples. Each entry multiple changes
the equity sizing requirement. At a distressed exit the signed residual can
be negative while common equity receives zero.

In [ ]:
entries = ["7.5", "8", "8.5"]
exits = ["8", "9", "10", "11"]
grid = pd.DataFrame(
    [[price_buyout(entry, exit).exit.sized_moic for exit in exits] for entry in entries],
    index=pd.Index(entries, name="Purchase EV / EBITDA"),
    columns=pd.Index(exits, name="Exit EV / EBITDA"),
)
assert grid.shape == (3, 4)
assert (grid.diff(axis=0).iloc[1:] < 0).all().all()
assert (grid.diff(axis=1).iloc[:, 1:] > 0).all().all()
distressed = price_buyout(exit_multiple="3", sponsor_commitment=usd(320))
assert distressed.exit.common_equity_residual.amount == -40
assert distressed.exit.sponsor_proceeds.amount == 0
display(grid)